# SMILES Validator Analysis

In [ ]:
import sys
sys.path.insert(0, '..')

from src import validate_smiles, parse_smiles
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from collections import Counter

# Validators
from rdkit import Chem
import partialsmiles as ps
import pysmiles
from molvs import validate_smiles as molvs_validate

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

: 

In [ ]:
# Define all validators
def validate_rdkit(mol: str) -> bool:
    try:
        return Chem.MolFromSmiles(mol) is not None
    except:
        return False

def validate_partialsmiles(mol: str) -> bool:
    try:
        ps.ParseSmiles(mol, partial=False)
        return True
    except:
        return False

def validate_pysmiles(mol: str) -> bool:
    try:
        pysmiles.read_smiles(mol)
        return True
    except:
        return False

def validate_molvs(mol: str) -> bool:
    try:
        errors = molvs_validate(mol)
        return len(errors) == 0
    except:
        return False

def validate_yacc(mol: str) -> bool:
    return validate_smiles(mol)[0]

VALIDATORS = {
    'YACC': validate_yacc,
    'RDKit': validate_rdkit,
    'PartialSMILES': validate_partialsmiles,
    'PySMILES': validate_pysmiles,
    'MolVS': validate_molvs,
}

## 1. Ground Truth Validation (Test Dataset)

In [ ]:
test_df = pd.read_csv('../tests/data/test_molecules_from_values.csv')
test_df = test_df.dropna(subset=['smiles'])
print(f"Test dataset: {len(test_df)} molecules ({test_df['is_valid'].sum()} valid, {(~test_df['is_valid']).sum()} invalid)")

In [ ]:
# Run all validators
for name, validator in VALIDATORS.items():
    test_df[name] = test_df['smiles'].apply(validator)
test_df['ground_truth'] = test_df['is_valid']

### Metrics Comparison

In [ ]:
def compute_metrics(y_true, y_pred, name):
    tp = ((y_true == True) & (y_pred == True)).sum()
    tn = ((y_true == False) & (y_pred == False)).sum()
    fp = ((y_true == False) & (y_pred == True)).sum()
    fn = ((y_true == True) & (y_pred == False)).sum()
    
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {'Validator': name, 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
            'Accuracy': f'{accuracy:.2%}', 'Precision': f'{precision:.2%}', 
            'Recall': f'{recall:.2%}', 'F1': f'{f1:.2%}'}

metrics = [compute_metrics(test_df['ground_truth'], test_df[name], name) for name in VALIDATORS]
metrics_df = pd.DataFrame(metrics).sort_values('Accuracy', ascending=False)
display(metrics_df)

In [ ]:
# Bar chart of F1 scores
fig, ax = plt.subplots(figsize=(10, 5))
f1_values = [float(m['F1'].strip('%')) for m in metrics]
colors = sns.color_palette('viridis', len(VALIDATORS))
bars = ax.bar([m['Validator'] for m in metrics], f1_values, color=colors)
ax.set_ylabel('F1 Score (%)')
ax.set_title('Validator F1 Scores on Ground Truth Dataset')
ax.set_ylim(0, 105)
for bar, val in zip(bars, f1_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}%', ha='center')
plt.tight_layout()
plt.savefig('f1_scores.png', dpi=150)
plt.show()

### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for ax, name in zip(axes, VALIDATORS.keys()):
    cm = pd.crosstab(test_df['ground_truth'], test_df[name], 
                     rownames=['Actual'], colnames=['Predicted'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(f'{name}')
    ax.set_xticklabels(['Invalid', 'Valid'])
    ax.set_yticklabels(['Invalid', 'Valid'])

axes[-1].axis('off')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150)
plt.show()

### False Positives Table

In [ ]:
fp_counts = {name: ((test_df['ground_truth'] == False) & (test_df[name] == True)).sum() for name in VALIDATORS}
fp_df = pd.DataFrame(list(fp_counts.items()), columns=['Validator', 'False Positives']).sort_values('False Positives')
display(fp_df)

print("\nFalse Positives by Validator:")
for name in VALIDATORS:
    fps = test_df[(test_df['ground_truth'] == False) & (test_df[name] == True)]['smiles'].tolist()
    if fps:
        print(f"\n{name} ({len(fps)}):")
        for s in fps[:5]:
            print(f"  {s[:60]}")

### False Negatives Table

In [ ]:
fn_counts = {name: ((test_df['ground_truth'] == True) & (test_df[name] == False)).sum() for name in VALIDATORS}
fn_df = pd.DataFrame(list(fn_counts.items()), columns=['Validator', 'False Negatives']).sort_values('False Negatives')
display(fn_df)

## 2. Large Dataset Benchmark

In [ ]:
def load_benchmark_data():
    files = glob('../data/*.csv')
    results = []
    for csv in files:
        try:
            df = pd.read_csv(csv)
        except UnicodeDecodeError:
            df = pd.read_csv(csv, compression='gzip')
        if 'smiles' not in df.columns:
            continue
        dataset_name = csv.split('/')[-1].replace('.csv', '')
        for smiles in df['smiles'].dropna():
            results.append({'dataset': dataset_name, 'smiles': smiles})
        print(f"Loaded {dataset_name}: {len(df)} molecules")
    return pd.DataFrame(results)

bench_df = load_benchmark_data()
print(f"\nTotal: {len(bench_df)} molecules")

In [ ]:
# Run all validators
for name, validator in VALIDATORS.items():
    print(f"Running {name}...")
    bench_df[name] = bench_df['smiles'].apply(validator)

### Validation Rates by Dataset

In [ ]:
summary = bench_df.groupby('dataset').agg(
    {**{'smiles': 'count'}, **{name: 'mean' for name in VALIDATORS}}
).rename(columns={'smiles': 'count'})
for name in VALIDATORS:
    summary[name] = (summary[name] * 100).round(2)
display(summary)

In [ ]:
# Grouped bar chart
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(summary))
width = 0.15
colors = sns.color_palette('viridis', len(VALIDATORS))

for i, name in enumerate(VALIDATORS):
    offset = (i - len(VALIDATORS)/2 + 0.5) * width
    ax.bar(x + offset, summary[name], width, label=name, color=colors[i])

ax.set_ylabel('Validation Rate (%)')
ax.set_title('Validation Rate by Dataset and Validator')
ax.set_xticks(x)
ax.set_xticklabels(summary.index, rotation=45)
ax.legend(loc='lower left')
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('validation_rates.png', dpi=150)
plt.show()

### Overall Pass Rates

In [ ]:
total = len(bench_df)
pass_rates = {name: bench_df[name].sum() / total * 100 for name in VALIDATORS}
pass_df = pd.DataFrame(list(pass_rates.items()), columns=['Validator', 'Pass Rate (%)']).sort_values('Pass Rate (%)', ascending=False)
pass_df['Pass Rate (%)'] = pass_df['Pass Rate (%)'].round(2)
display(pass_df)

In [ ]:
# Horizontal bar chart of pass rates
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette('viridis', len(pass_df))
ax.barh(pass_df['Validator'], pass_df['Pass Rate (%)'], color=colors)
ax.set_xlabel('Pass Rate (%)')
ax.set_title('Overall Validation Pass Rate (108K molecules)')
ax.set_xlim(0, 105)
for i, val in enumerate(pass_df['Pass Rate (%)']):
    ax.text(val + 1, i, f'{val:.1f}%', va='center')
plt.tight_layout()
plt.savefig('pass_rates.png', dpi=150)
plt.show()

### Agreement Heatmap

In [ ]:
# Pairwise agreement between validators
agreement_matrix = pd.DataFrame(index=VALIDATORS.keys(), columns=VALIDATORS.keys(), dtype=float)
for v1 in VALIDATORS:
    for v2 in VALIDATORS:
        agreement = (bench_df[v1] == bench_df[v2]).mean() * 100
        agreement_matrix.loc[v1, v2] = agreement

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(agreement_matrix.astype(float), annot=True, fmt='.1f', cmap='RdYlGn', ax=ax, vmin=70, vmax=100)
ax.set_title('Pairwise Agreement Between Validators (%)')
plt.tight_layout()
plt.savefig('agreement_heatmap.png', dpi=150)
plt.show()

### YACC Rejection Reasons

In [ ]:
yacc_rejects = bench_df[bench_df['YACC'] == False]['smiles'].head(500)

failure_reasons = []
for smiles in yacc_rejects:
    _, err = validate_smiles(smiles)
    if err:
        err_str = str(err)
        if 'validate_smiles' in err_str:
            failure_reasons.append('Trailing C Check')
        elif 'validate_rings_and_valency' in err_str:
            failure_reasons.append('Valency Check')
        elif 'validate_aromaticity' in err_str:
            failure_reasons.append('Aromaticity Check')
        elif 'chain' in err_str or 'Parser' in err_str:
            failure_reasons.append('Parsing Error')
        else:
            failure_reasons.append('Other')

reason_df = pd.DataFrame(Counter(failure_reasons).items(), columns=['Reason', 'Count']).sort_values('Count', ascending=False)
reason_df['%'] = (reason_df['Count'] / reason_df['Count'].sum() * 100).round(1)
display(reason_df)

## 3. Summary

In [ ]:
print("="*60)
print("SUMMARY")
print("="*60)
print(f"Ground truth test set: {len(test_df)} molecules")
print(f"Benchmark dataset: {len(bench_df):,} molecules")
print()
print("Ground Truth Performance (F1 Score):")
for m in sorted(metrics, key=lambda x: -float(x['F1'].strip('%'))):
    print(f"  {m['Validator']:15} {m['F1']}")
print()
print("Benchmark Pass Rates:")
for _, row in pass_df.iterrows():
    print(f"  {row['Validator']:15} {row['Pass Rate (%)']:.2f}%")